# Notebook 03 — 1D U-Net CNN Training

Goals:
1. Sanity-check: overfit on 10 samples → loss should reach near-zero
2. Full training run with early stopping
3. Plot train / val loss curves and val F1 per epoch
4. Load best checkpoint and evaluate on test set
5. Visual inspection of predictions on 6 test examples

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import csv
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset

from src.dataset  import build_datasets
from src.models   import AxleUNet
from src.train    import train, train_one_epoch, val_loss_epoch, build_loss
from src.evaluate import evaluate_model, print_metrics, pulses_to_peaks

JSON_PATH  = '../axle_data.json/axle_data.json'
CKPT_DIR   = '../checkpoints'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED       = 42
print(f'Using device: {DEVICE}')

Using device: cpu


## 1. Sanity check — overfit on 10 samples

In [3]:
torch.manual_seed(SEED)
train_ds, val_ds, test_ds = build_datasets(JSON_PATH, seed=SEED)

tiny_ds     = Subset(train_ds, list(range(10)))
tiny_loader = DataLoader(tiny_ds, batch_size=10, shuffle=False)

tiny_model = AxleUNet().to(DEVICE)
# Use plain BCE (no pos_weight) for the sanity-check so loss scale is 0→1
criterion_plain = torch.nn.BCEWithLogitsLoss()
optimizer  = torch.optim.AdamW(tiny_model.parameters(), lr=5e-3)

for epoch in range(1, 201):
    loss = train_one_epoch(tiny_model, tiny_loader, criterion_plain, optimizer, DEVICE, scaler=None)
    if epoch % 40 == 0:
        print(f'Epoch {epoch:3d}  loss={loss:.6f}')

print('✓ Sanity check PASSED' if loss < 0.01 else f'WARNING: loss={loss:.4f} (model may need more epochs or tuning)')

Loaded 32,141 records from axle_data.json
  Train: 25,712  |  Val: 3,214  |  Test: 3,215
  Signal stats (train) — mean: 0.0322, std: 0.1866


Epoch  40  loss=0.056547


Epoch  80  loss=0.019642


Epoch 120  loss=0.013314


Epoch 160  loss=0.008959


Epoch 200  loss=0.006162
✓ Sanity check PASSED


## 2. Full training run

In [ ]:
# NOTE: On CPU, each epoch takes ~5-15 min.
# This cell runs 5 epochs as a pipeline demo.
# For the full 50-epoch run, use the CLI:
#   python -m src.train --model cnn --epochs 50 --batch_size 64 --patience 10
N_DEMO_EPOCHS = 5

test_metrics = train(
    model_name     = 'cnn',
    json_path      = JSON_PATH,
    epochs         = N_DEMO_EPOCHS,
    batch_size     = 64,
    lr             = 1e-3,
    patience       = 10,
    num_workers    = 0,
    seed           = SEED,
    checkpoint_dir = CKPT_DIR,
)
print_metrics(test_metrics, prefix='CNN demo test')

Device : cpu
Model  : CNN
Loaded 32,141 records from axle_data.json
  Train: 25,712  |  Val: 3,214  |  Test: 3,215
  Signal stats (train) — mean: 0.0322, std: 0.1866
Params : 2,620,289


  pos_weight = 286.07  (positives: 116437 / 33425600 = 0.348%)


Epoch   1/5 | train_loss=0.20099  val_loss=0.05537 | [val] Precision: 0.9893  Recall: 0.9997  F1: 0.9945  MATE: 0.24 samples  (TP=14576, FP=157, FN=5)
  lr=1.00e-03  time=599.6s
  ✓ Saved best checkpoint (val F1=0.9945)


Epoch   2/5 | train_loss=0.05367  val_loss=0.18899 | [val] Precision: 0.9955  Recall: 0.9990  F1: 0.9972  MATE: 0.62 samples  (TP=14566, FP=66, FN=15)
  lr=1.00e-03  time=3423.1s
  ✓ Saved best checkpoint (val F1=0.9972)


  train:  13%|█▎        | 53/402 [01:59<16:31,  2.84s/it]

## 3. Plot training curves

In [ ]:
log_path = os.path.join(CKPT_DIR, 'cnn_log.csv')
epochs_list, train_losses, val_losses, val_f1s = [], [], [], []

with open(log_path) as f:
    reader = csv.DictReader(f)
    for row in reader:
        epochs_list.append(int(row['epoch']))
        train_losses.append(float(row['train_loss']))
        val_losses.append(float(row['val_loss']))
        val_f1s.append(float(row['val_f1']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_list, train_losses, label='train loss')
ax1.plot(epochs_list, val_losses,   label='val loss')
ax1.set_xlabel('Epoch');  ax1.set_ylabel('BCE Loss')
ax1.set_title('CNN — Loss curves');  ax1.legend()

ax2.plot(epochs_list, val_f1s, color='green', marker='o', markersize=2)
ax2.set_xlabel('Epoch');  ax2.set_ylabel('Val F1')
ax2.set_title('CNN — Validation F1')

plt.tight_layout()
plt.show()

## 4. Visual inspection on 6 test examples

In [ ]:
# Reload best model
ckpt  = torch.load(os.path.join(CKPT_DIR, 'cnn_best.pt'), map_location=DEVICE)
model = AxleUNet().to(DEVICE)
model.load_state_dict(ckpt['state_dict'])
model.eval()

np.random.seed(SEED)
sample_idx = np.random.choice(len(test_ds), 6, replace=False)

fig, axes = plt.subplots(6, 1, figsize=(14, 3 * 6))
for ax, idx in zip(axes, sample_idx):
    sig_t, pulse_t = test_ds[idx]
    with torch.no_grad():
        logit = model(sig_t.unsqueeze(0).to(DEVICE))
        prob  = torch.sigmoid(logit).squeeze().cpu().numpy()

    sig         = sig_t.squeeze(0).numpy()
    true        = pulse_t.numpy()
    true_peaks  = pulses_to_peaks(true, threshold=0.5)
    pred_peaks  = pulses_to_peaks(prob, threshold=0.5)

    ax.plot(sig,  color='steelblue', linewidth=0.8,  label='signal')
    ax.plot(prob * (sig.max() - sig.min()) + sig.min(),
            color='orange', linewidth=0.8, alpha=0.7, label='CNN prob (scaled)')
    ax.vlines(true_peaks, sig.min(), sig.max(), colors='green',
              linestyles='-', linewidth=1.2, label=f'true ({len(true_peaks)})')
    ax.vlines(pred_peaks, sig.min(), sig.max(), colors='red',
              linestyles='--', linewidth=1.0, label=f'pred ({len(pred_peaks)})')
    ax.set_title(f'Test record {idx}')
    ax.legend(loc='upper right', fontsize=7)

plt.tight_layout()
plt.show()